## Embeddings (dense ViT → patch + cls token parquet)

Stream chips with **per-chip view batching**, write `ssl4eo_cls-patch.parquet`.

In [ ]:
import hashlib
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import gee
import model_library
import tile_utils

%load_ext autoreload
%autoreload 2

data_timestamp = "2026-03-24T23:15"
data_dir = Path(f"../data/training_patches{data_timestamp}")


In [ ]:
# Geo Foundation Model and structure

VIEWPORT = 224 # ViT input image width
PATCH_PX = 16  # ViT patch width 
PATCH_CT = 14  # ViT patch count in width or height

inference_config = gee.InferenceConfig(
    embedding_strategy="cls_patch",
    embedding_batch_size=16,  # >= train_n_views so one chip's views stay one GPU batch
    embed_model_name = "ssl4eo_vit_s16",
    embed_model_path = "../models/SSL4EO/pretrained/ssl4eo_vit_small_patch16_weights.pth",
    embed_model_chip_size = VIEWPORT,
    geo_chip_size = VIEWPORT
)

assert VIEWPORT == PATCH_CT * PATCH_PX, (
    "ViT geometry: viewport must equal patch_ct * patch_px (match model + jitter grid)"
)


In [ ]:
cl_path = data_dir.parent / "sampling_locations" / f"collected_locations{data_timestamp}.geojson"
collected_locations = gpd.read_file(cl_path)
print("Input labeled datasets:")
set(collected_locations.source_file)

### Embedding inference

Uses ``InferenceEngine.embed_dense`` with **batched viewports per super-chip** (see ``cls_patch_feature_batch`` / ``collect_cls_patch_embedding_table``). Keep ``embedding_batch_size`` ≥ ``train_n_views``.

``InferenceEngine`` still carries GEE + Keras classifier; this notebook only uses the ViT path.


In [ ]:
def raster_center_point(raster_src):
    """Bounds midpoint as a Point; training GeoTIFFs are EPSG:4326 (repo convention)."""
    from shapely.geometry import Point

    bounds = raster_src.bounds
    return Point((bounds.left + bounds.right) / 2.0, (bounds.bottom + bounds.top) / 2.0)


def load_chip(path, bands_to_use=None):
    """Load one GeoTIFF chip as float32 HWC and its EPSG:4326 label-center geometry."""
    path = Path(path)
    with rasterio.open(path) as raster_src:
        chip_center = raster_center_point(raster_src)
        chip_array = raster_src.read()
        if bands_to_use is not None:
            chip_array = chip_array[np.array(bands_to_use, dtype=np.intp), :, :]
        chip_array = np.moveaxis(chip_array, 0, -1)
        chip_array = chip_array.astype(np.float32) / 10000.0
    return chip_array, chip_center


def load_dataset(data_dir, splits=None, source_files=None, bands_to_use=None):
    """
    Generator: yield one on-disk chip at a time to avoid holding the full dataset in RAM.

    Layout: data_dir / <source_stem> / {split} / {0|1} / *.tif

    Yields
    ------
    chip_hwc : np.ndarray
        Image (H, W, C), float32.
    label : int
        0 or 1 from parent folder name.
    split : str
        Split folder name (e.g. train, val).
    chip_path_posix : str
        Path to the .tif.
    chip_center : shapely Point
        Chip center in EPSG:4326.

    source_files: optional iterable of source folder names or paths; stems must match
        directories under data_dir. None = all top-level source dirs.

    If nothing matches, the generator yields no items (see collect_cls_patch_embedding_table warning).
    """
    root = Path(data_dir)
    if isinstance(splits, str):
        raise TypeError(
            "splits must be a list/tuple of split folder names (e.g. ['train']), "
            "not a single str — iterating a str splits per-character and yields almost no files."
        )
    allowed_source_stems = (
        {Path(s).stem for s in source_files} if source_files is not None else None
    )

    def iter_source_dirs():
        if allowed_source_stems is None:
            for child in sorted(p for p in root.iterdir() if p.is_dir()):
                yield child.name, child
        else:
            for source_stem in sorted(allowed_source_stems):
                source_dir = root / source_stem
                if source_dir.is_dir():
                    yield source_stem, source_dir

    if splits is None:
        split_names_found = set()
        for source_stem, source_dir in iter_source_dirs():
            for tif_path in source_dir.glob("**/[01]/*.tif"):
                if tif_path.parent.name not in {"0", "1"}:
                    continue
                split_names_found.add(tif_path.parent.parent.name)
        use_splits = sorted(split_names_found)
    else:
        use_splits = list(splits)

    for split_name in use_splits:
        saw_any_tif = False
        for source_stem, source_dir in iter_source_dirs():
            for class_subdir, class_label in (("0", 0), ("1", 1)):
                for tif_path in sorted(source_dir.glob(f"{split_name}/{class_subdir}/*.tif")):
                    saw_any_tif = True
                    chip_hwc, chip_center = load_chip(tif_path, bands_to_use)
                    chip_path_posix = tif_path.as_posix()
                    yield chip_hwc, class_label, split_name, chip_path_posix, chip_center
        if not saw_any_tif:
            print(f"No files found for split={split_name!r}; skipping.")


def quantize(embeddings, lower_bound=-5, upper_bound=5):
    clipped = np.clip(embeddings, lower_bound, upper_bound)
    normalized = (clipped - lower_bound) / (upper_bound - lower_bound)
    scaled = normalized * 255
    return scaled.astype(np.uint8)


def get_deterministic_jitter(path_key, view_index=0, *, patch_ct=PATCH_CT):
    """
    Reproducible ViT patch grid indices (row, col) in [0, patch_ct).

    Includes ``view_index`` in the hash so ``n_views > 1`` with jitter='deterministic'
    yields distinct windows per view (path alone would repeat the same offset).
    """
    key = f"{path_key}|view{view_index}"
    hash_hex = hashlib.md5(key.encode()).hexdigest()
    grid_row = int(hash_hex[:8], 16) % patch_ct
    grid_col = int(hash_hex[8:16], 16) % patch_ct
    return grid_row, grid_col


def get_seeded_random_jitter(path_key, view_index, *, patch_ct=PATCH_CT):
    """
    Reproducible (row, col) patch indices for this path and view_index.

    Uses an isolated ``np.random.RandomState`` per call (no global RNG pollution).
    """
    seed_str = f"{path_key}_{view_index}"
    seed = int(hashlib.md5(seed_str.encode()).hexdigest(), 16) % (2**32)
    rng = np.random.RandomState(seed)
    grid_col = rng.randint(0, patch_ct)
    grid_row = rng.randint(0, patch_ct)
    return grid_row, grid_col


def extract_jittered_viewport(
    super_chip, patch_loc, *, viewport=VIEWPORT, patch_px=PATCH_PX
):
    """Crop a viewport×viewport window so the label center falls in ViT patch (row, col).

    ``patch_loc`` is (row_i, col_j) in the patch_ct×patch_ct grid (0-based), matching ``embed_dense`` spatial layout.
    ``super_chip`` is (nrows, ncols, bands).
    """
    row_i, col_j = patch_loc
    nrows, ncols = super_chip.shape[0], super_chip.shape[1]
    row_center = nrows // 2
    col_center = ncols // 2
    half_patch = patch_px // 2
    start_row = row_center + half_patch - (row_i + 1) * patch_px
    start_col = col_center + half_patch - (col_j + 1) * patch_px

    if start_row < 0 or start_col < 0 or start_row + viewport > nrows or start_col + viewport > ncols:
        raise ValueError(
            f"Viewport out of bounds: super_chip {nrows}x{ncols}, "
            f"start=({start_row},{start_col}), patch_loc={patch_loc!r}"
        )

    viewport_hwc = super_chip[start_row : start_row + viewport, start_col : start_col + viewport]
    if viewport_hwc.shape[0] != viewport or viewport_hwc.shape[1] != viewport:
        raise RuntimeError(f"Expected ({viewport},{viewport}) crop, got {viewport_hwc.shape[:2]}")
    return viewport_hwc


def iter_viewports_for_chip(
    super_chip,
    path_key,
    *,
    n_views,
    jitter,
    viewport=VIEWPORT,
    patch_px=PATCH_PX,
    patch_ct=PATCH_CT,
):
    """Yield (view_index, viewport_hwc, patch_loc) for one on-disk super-chip.

    ``path_key`` is hashed for reproducible jitter (use the chip path string).
    """
    for view_index in range(n_views):
        if jitter == "deterministic":
            patch_loc = get_deterministic_jitter(
                path_key, view_index=view_index, patch_ct=patch_ct
            )
        elif jitter == "seeded":
            patch_loc = get_seeded_random_jitter(path_key, view_index, patch_ct=patch_ct)
        else:
            raise ValueError(f"jitter must be 'deterministic' or 'seeded', got {jitter!r}")
        viewport_hwc = extract_jittered_viewport(
            super_chip, patch_loc, viewport=viewport, patch_px=patch_px
        )
        yield view_index, viewport_hwc, patch_loc


def extract_target_patches(spatial_emb, patch_locs, *, patch_ct=PATCH_CT):
    """
    spatial_emb: (B, patch_ct, patch_ct, Dim)
    patch_locs: (B, 2) with (row_i, col_j) per batch row
    """
    grid_height, grid_width = spatial_emb.shape[1], spatial_emb.shape[2]
    if grid_height != patch_ct or grid_width != patch_ct:
        raise ValueError(
            f"spatial_emb grid {grid_height}x{grid_width} expected {patch_ct}x{patch_ct} (set patch_ct=... to match model)"
        )
    patch_locs = np.asarray(patch_locs, dtype=np.intp)
    batch_indices = np.arange(len(spatial_emb), dtype=np.intp)
    rows = patch_locs[:, 0]
    cols = patch_locs[:, 1]
    return spatial_emb[batch_indices, rows, cols, :]


def cls_patch_feature_batch(
    engine,
    viewports_bhwc,
    chip_geom,
    patch_locs_batch,
    *,
    feature_col_names,
    quantized=False,
    patch_ct=PATCH_CT,
):
    """Run ``embed_dense`` once for all jittered viewports of one super-chip.

    Parameters
    ----------
    viewports_bhwc : array (B, H, W, C)
    chip_geom : shapely geometry (repeated B times for the engine API)
    patch_locs_batch : array (B, 2) int — ViT patch grid indices per view

    Returns
    -------
    (B, D) float (or uint8 if ``quantized``) feature matrix (cls + one spatial patch each).
    """
    viewports_batch = np.asarray(viewports_bhwc, dtype=np.float32)
    if viewports_batch.ndim != 4:
        raise ValueError(f"expected viewports (B,H,W,C), got shape {viewports_batch.shape}")
    num_views = viewports_batch.shape[0]
    patch_locs_arr = np.asarray(patch_locs_batch, dtype=np.intp)
    if patch_locs_arr.shape != (num_views, 2):
        raise ValueError(
            f"patch_locs_batch shape {patch_locs_arr.shape}, expected ({num_views}, 2)"
        )

    chip_geoms_geodataframe = gpd.GeoDataFrame(
        geometry=[chip_geom] * num_views, crs="EPSG:4326"
    )
    emb_dict = engine.embed_dense(viewports_batch, chip_geoms_geodataframe, tile=None)
    spatial = emb_dict["spatial"]
    if spatial.shape[1] != patch_ct or spatial.shape[2] != patch_ct:
        raise ValueError(
            f"embed_dense spatial shape {spatial.shape[1:3]} != patch_ct {patch_ct}x{patch_ct}"
        )
    patch_token_embeddings = extract_target_patches(
        spatial, patch_locs_arr, patch_ct=patch_ct
    )
    cls_and_patch = np.concatenate([emb_dict["cls"], patch_token_embeddings], axis=1)
    expected_width = len(feature_col_names)
    if cls_and_patch.shape[1] != expected_width:
        raise ValueError(
            f"Embedding width {cls_and_patch.shape[1]} != number of columns {expected_width}"
        )
    return quantize(cls_and_patch) if quantized else cls_and_patch


def collect_cls_patch_embedding_table(
    data_dir,
    engine,
    *,
    splits=None,
    source_files=None,
    bands_to_use=None,
    train_n_views=5,
    train_jitter="seeded",
    eval_n_views=1,
    eval_jitter="deterministic",
    quantized=False,
    feature_cols=None,
    patch_ct=PATCH_CT,
    progress_every=500,
):
    """Stream super-chips from disk; batch all views per chip through the ViT once.

    Eval splits usually have ``eval_n_views == 1`` (same as batch-1). Train uses
    ``train_n_views`` jittered crops in a single ``embed_dense`` call per chip.

    ``InferenceConfig.embedding_batch_size`` should be >= ``train_n_views`` so the
    engine does not split one chip's views across multiple GPU sub-batches.

    If ``feature_cols`` is None and nothing is embedded, ``feature_columns`` from the
    next cell must already exist for the empty schema.
    """
    embedding_column_names = feature_columns if feature_cols is None else feature_cols
    output_rows = []
    chip_stream = load_dataset(
        data_dir,
        splits=splits,
        source_files=source_files,
        bands_to_use=bands_to_use,
    )
    for chip_index, (super_chip, label, split_name, chip_path_posix, chip_center_geom) in enumerate(
        chip_stream
    ):
        if progress_every and (chip_index + 1) % progress_every == 0:
            print(f"Embedded {chip_index + 1} chips (latest split={split_name!r})...")
        n_views, jitter_mode = (
            (train_n_views, train_jitter)
            if split_name == "train"
            else (eval_n_views, eval_jitter)
        )
        viewport_rasters = []
        view_indices = []
        jitter_patch_locations = []
        for view_index, viewport_hwc, patch_loc in iter_viewports_for_chip(
            super_chip,
            chip_path_posix,
            n_views=n_views,
            jitter=jitter_mode,
            patch_ct=patch_ct,
        ):
            viewport_rasters.append(viewport_hwc)
            view_indices.append(view_index)
            jitter_patch_locations.append(patch_loc)
        if not viewport_rasters:
            continue
        viewports_batch = np.stack(viewport_rasters, axis=0)
        patch_locs_batch = np.array(jitter_patch_locations, dtype=np.intp)
        cls_spatial_features_batch = cls_patch_feature_batch(
            engine,
            viewports_batch,
            chip_center_geom,
            patch_locs_batch,
            feature_col_names=embedding_column_names,
            quantized=quantized,
            patch_ct=patch_ct,
        )
        for view_row_index, view_index in enumerate(view_indices):
            row_dict = {
                "label": label,
                "split": split_name,
                "path": chip_path_posix,
                "view": view_index,
                "patch": jitter_patch_locations[view_row_index],
                "geometry": chip_center_geom,
            }
            for column_name, value in zip(
                embedding_column_names, cls_spatial_features_batch[view_row_index]
            ):
                row_dict[column_name] = value
            output_rows.append(row_dict)

    if not output_rows:
        allowed_repr = (
            None if source_files is None else {Path(s).stem for s in source_files}
        )
        print(
            f"Warning: no .tif chips embedded under {Path(data_dir).resolve()!r} "
            f"splits={splits!r} (source filter={allowed_repr!r})."
        )
        return gpd.GeoDataFrame(
            {
                "label": pd.Series([], dtype=np.int64),
                "split": pd.Series([], dtype=object),
                "path": pd.Series([], dtype=object),
                "view": pd.Series([], dtype=np.int64),
                "patch": pd.Series([], dtype=object),
                **{
                    c: pd.Series([], dtype=np.float32)
                    for c in embedding_column_names
                },
            },
            geometry=gpd.GeoSeries([], crs="EPSG:4326"),
        )

    return gpd.GeoDataFrame(output_rows, crs="EPSG:4326")


In [ ]:
embedding_engine = gee.InferenceEngine(
    "2000-01-01",  # Dates only apply to GEE data extractor, unused in this notebook
    "2000-01-02",
    gee.DataConfig(),
    inference_config,
)
print(f"Device: {embedding_engine.device}")

output_dim = embedding_engine.embed_model.norm.normalized_shape[0]
feature_columns = [f"{c}{i}" for c in ['cls', 'spatial'] for i in range(output_dim)]

quantized = False

In [ ]:
stems = {Path(p).stem for p in collected_locations["source_file"] if "ACA" in str(p)}
source_files = [data_dir.parent / "sampling_locations" / f"{s}.geojson" for s in stems]
source_files

In [ ]:
# Optional: restrict to some GeoJSON sources (stems must match folders under data_dir).
# example:
# stems = {Path(p).stem for p in collected_locations["source_file"] if "random" in str(p)}
# source_files = [data_dir.parent / "sampling_locations" / f"{s}.geojson" for s in stems]  

splits_to_embed = ["val", "test1", "test2", "train"]

df = collect_cls_patch_embedding_table(
    data_dir,
    embedding_engine,
    splits=splits_to_embed,
    source_files=source_files,#None,  # or a list of paths / stems for a subset
    train_n_views=10,
    train_jitter="seeded",
    eval_n_views=1,
    eval_jitter="deterministic",
    quantized=quantized,
    progress_every=100,
)


In [ ]:
df[["label", "split"]].value_counts()

In [ ]:
parquet_path = data_dir / "ssl4eo_cls-patch.parquet"
df.to_parquet(parquet_path, index=False)
print(f"Wrote {len(df)} rows to {parquet_path}")